# Mock OpenAI Calls with the SDK

This notebook uses the `openai` JavaScript package against a local, OpenAI-compatible route served by a cell in this kernel. The package is loaded through JavaScript kernel magic imports, so the first run needs access to the configured package CDN. No real API key is used, and the API calls do not contact OpenAI.

Run the cells from top to bottom in a JavaScript (IFrame) kernel.

In [ ]:
import OpenAI from 'openai';

## Define the mock endpoint

The client uses the OpenAI SDK with a notebook-specific local route. A custom `fetch` adapter rewrites the SDK's absolute API URL to a same-origin path, which also works when the IFrame runtime has an opaque origin. The handler returns either a Chat Completions response or Server-Sent Events, depending on the request's `stream` option.

In [ ]:
const routePrefix = '/api/openai-sdk-mock/';
claimPrefix(routePrefix);

app.post(`${routePrefix}v1/chat/completions`, async c => {
  const request = await c.req.json();
  const userMessage = request.messages?.find(message => message.role === 'user');
  const prompt = typeof userMessage?.content === 'string' ? userMessage.content : 'your message';
  const model = request.model ?? 'gpt-4o-mini';
  const created = Math.floor(Date.now() / 1000);
  const id = 'chatcmpl-openai-sdk-mock';
  const assistantText = `This is a mocked response. I received: ${prompt}`;

  if (request.stream === true) {
    const textChunks = assistantText.match(/[\s\S]{1,12}/gu) ?? [assistantText];
    const makeChunk = choices => ({ id, object: 'chat.completion.chunk', created, model, choices });
    const frames = [
      makeChunk([{ index: 0, delta: { role: 'assistant' }, finish_reason: null }]),
      ...textChunks.map(content =>
        makeChunk([{ index: 0, delta: { content }, finish_reason: null }])
      ),
      makeChunk([{ index: 0, delta: {}, finish_reason: 'stop' }])
    ];
    const encoder = new TextEncoder();
    const body = new ReadableStream({
      start(controller) {
        for (const frame of frames) {
          controller.enqueue(encoder.encode(`data: ${JSON.stringify(frame)}\n\n`));
        }
        controller.enqueue(encoder.encode('data: [DONE]\n\n'));
        controller.close();
      }
    });

    return c.body(body, 200, {
      'content-type': 'text/event-stream; charset=utf-8',
      'cache-control': 'no-cache'
    });
  }

  return c.json({
    id,
    object: 'chat.completion',
    created,
    model,
    choices: [
      {
        index: 0,
        message: { role: 'assistant', content: assistantText },
        finish_reason: 'stop'
      }
    ],
    usage: { prompt_tokens: 0, completion_tokens: 0, total_tokens: 0 }
  });
});

const localOpenAIFetch = (input, init) => {
  const requestUrl = new URL(input instanceof Request ? input.url : String(input));
  if (!requestUrl.pathname.startsWith('/v1/')) {
    throw new Error(`Unexpected OpenAI SDK path: ${requestUrl.pathname}`);
  }
  const localPath = `${routePrefix}v1${requestUrl.pathname.slice('/v1'.length)}${requestUrl.search}`;
  return fetch(localPath, init);
};

const client = new OpenAI({
  apiKey: 'mock-api-key',
  baseURL: 'https://api.openai.com/v1',
  dangerouslyAllowBrowser: true,
  fetch: localOpenAIFetch
});

Confirm that this kernel owns the route prefix.

In [ ]:
await claimedPrefixes()

## Make a Chat Completions call

The SDK sends its normal Chat Completions request to the local `baseURL`; the cell-defined route returns the mock response.

In [ ]:
const completion = await client.chat.completions.create({
  model: 'gpt-4o-mini',
  messages: [{ role: 'user', content: 'Explain cell-defined routes in one sentence.' }]
});

console.log(completion.choices[0].message.content);
completion

## Make a streaming call

With `stream: true`, the SDK exposes response chunks as an async iterable. Each content delta is printed with a one-second pause between SDK chunks.

In [ ]:
const stream = await client.chat.completions.create({
  model: 'gpt-4o-mini',
  messages: [{ role: 'user', content: 'What is streaming mode?' }],
  stream: true
});

let streamedText = '';
for await (const chunk of stream) {
  const content = chunk.choices[0]?.delta?.content;
  if (content) {
    streamedText += content;
    console.log(content);
  }
  await new Promise(resolve => setTimeout(resolve, 1000));
}

console.log(`Full response: ${streamedText}`);

Both calls use the OpenAI SDK, but the custom `baseURL` points to this kernel's local route. The current cell-route bridge buffers response bodies before forwarding them, so the streaming call demonstrates the SDK's streaming API and SSE format but not incremental end-to-end delivery.